In [ ]:
!pip install --no-index --find-links=/kaggle/input/datasets/eman3000/zarr-wheels zarr
# %% [code] — Official Submission Generator: Multi-Modal Fused Tracker
import os
import glob
import time
import numpy as np
import pandas as pd
import zarr
import cupy as cp
import cupyx.scipy.ndimage as cundi
from scipy.spatial import KDTree
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from scipy.sparse.csgraph import connected_components
import networkx as nx
from tqdm.auto import tqdm

# --- 1. Environment Paths & Constants ---
test_dir = "/kaggle/input/competitions/biohub-cell-tracking-during-development/test"
if not os.path.isdir(test_dir):
    test_dir = "/kaggle/input/biohub-cell-tracking-during-development/test"

output_csv = "/kaggle/working/submission.csv"

# Universal physical scale in microns (z, y, x)
SCALE_ZYX = np.array([1.625, 0.40625, 0.40625], dtype=np.float32)

# Tuned Detector Parameters
DOG_RADIUS_UM = 2.5
MERGE_RADIUS_UM = 6.0
MIN_DISTANCE_VOX = 3
THRESH_REL = 0.03 #3% or lower
GAMMA = 1.5
STRUCT_SIZE = (3, max(3, int(np.ceil(5.0 / SCALE_ZYX[1]))), max(3, int(np.ceil(5.0 / SCALE_ZYX[1]))))

# Tuned Tracker Parameters
MATCH_DIST_UM = 7.0
MAX_LOST_FRAMES = 2
MIN_HITS = 2

# Probabilistic Cost Normalization Parameters
LAMBDA_INT = 0.15
LAMBDA_GEOM = 0.10
SIGMA_SPATIAL = 2.5
SIGMA_INT = 0.05
SIGMA_GEOM = 0.08


# --- 2. Multi-Modal GPU Detector ---

def peak_local_max_gpu(vol_gpu: cp.ndarray, min_distance: int, threshold_rel: float) -> cp.ndarray:
    size = 2 * min_distance + 1
    local_max = cundi.maximum_filter(vol_gpu, size=size, mode="constant", cval=-cp.inf)
    thresh_val = threshold_rel * vol_gpu.max()
    mask = (vol_gpu == local_max) & (vol_gpu > thresh_val)
    return cp.argwhere(mask)


def detect_frame(raw_frame: np.ndarray) -> dict:
    raw_gpu = cp.asarray(raw_frame.astype(np.float32))

    s_xy = DOG_RADIUS_UM / (1.414 * SCALE_ZYX[1])
    s_z = DOG_RADIUS_UM / (1.414 * SCALE_ZYX[0])

    dog = cundi.gaussian_filter(raw_gpu, sigma=(s_z, s_xy, s_xy)) - \
          cundi.gaussian_filter(raw_gpu, sigma=(s_z * 1.6, s_xy * 1.6, s_xy * 1.6))
    dog_norm = cp.clip(dog, 0, None) / (cp.clip(dog, 0, None).max() + 1e-8)

    bg_local = cundi.minimum_filter(raw_gpu, size=STRUCT_SIZE)
    tophat = cp.clip(raw_gpu - bg_local, 0, None)
    local_max = cundi.maximum_filter(tophat, size=STRUCT_SIZE)
    local_norm = cp.clip(tophat / (local_max + 1e-6), 0.0, 1.0)

    score = dog_norm * (local_norm ** GAMMA)
    score = score / (score.max() + 1e-8)

    p_raw = cp.asnumpy(peak_local_max_gpu(score, min_distance=MIN_DISTANCE_VOX, threshold_rel=THRESH_REL))
    dog_cpu = cp.asnumpy(dog)

    del raw_gpu, dog, dog_norm, bg_local, tophat, local_max, local_norm, score

    if len(p_raw) == 0:
        del dog_cpu
        return {"coords": np.empty((0, 3)), "features": np.empty((0, 4))}

    Z, Y, X = raw_frame.shape
    rz, rxy = max(2, int(np.ceil(5.0 / SCALE_ZYX[0]))), max(4, int(np.ceil(5.0 / SCALE_ZYX[1])))
    features_list = []

    for vz, vy, vx in p_raw:
        z0, z1 = max(0, vz - rz), min(Z, vz + rz + 1)
        y0, y1 = max(0, vy - rxy), min(Y, vy + rxy + 1)
        x0, x1 = max(0, vx - rxy), min(X, vx + rxy + 1)

        raw_patch = raw_frame[z0:z1, y0:y1, x0:x1].astype(np.float32)
        dog_patch = dog_cpu[z0:z1, y0:y1, x0:x1]

        features_list.append(extract_cell_multimodal_features(raw_patch, dog_patch))

    del dog_cpu
    features_raw = np.array(features_list, dtype=np.float32)
    pts_raw = p_raw * SCALE_ZYX

    pts_merged, feats_merged = merge_in_radius_with_features(pts_raw, features_raw, radius_um=MERGE_RADIUS_UM)

    return {
        "coords": pts_merged,
        "features": feats_merged
    }

def extract_cell_multimodal_features(raw_patch, dog_patch):
    peak_dog = np.max(dog_patch)
    if peak_dog <= 0:
        return np.array([0.0, 1.0, 1.0, 1.0], dtype=np.float32)

    mask = (dog_patch > 0.15 * peak_dog) & (dog_patch > 0)
    n_vox = np.sum(mask)
    if n_vox < 8:
        return np.array([float(np.mean(raw_patch)), 1.0, 1.0, 1.0], dtype=np.float32)

    coords_um = np.argwhere(mask).astype(np.float64) * SCALE_ZYX
    weights = raw_patch[mask]
    total_w = np.sum(weights) + 1e-6

    # 1. Mean Intensity inside DoG border
    mean_int = float(total_w / n_vox)

    # 2. 3D Inertia Covariance Tensor in physical microns
    centroid_um = np.sum(coords_um * weights[:, None], axis=0) / total_w
    d_coords = coords_um - centroid_um
    cov_3d = (d_coords.T @ (d_coords * weights[:, None])) / total_w

    eigvals = np.linalg.eigvalsh(cov_3d)
    eigvals = np.sort(np.maximum(eigvals, 1e-6))[::-1]
    a, b, c = np.sqrt(eigvals[0]), np.sqrt(eigvals[1]), np.sqrt(eigvals[2])

    return np.array([mean_int, float(a), float(b), float(c)], dtype=np.float32)
    
def merge_in_radius_with_features(coords_um: np.ndarray, features: np.ndarray, radius_um: float = MERGE_RADIUS_UM):
    if len(coords_um) <= 1:
        return coords_um, features
    tree = KDTree(coords_um)
    adj = tree.sparse_distance_matrix(tree, max_distance=radius_um)
    n_comp, labels = connected_components(adj)
    
    merged_coords = np.array([coords_um[labels == k].mean(axis=0) for k in range(n_comp)])
    merged_feats = np.array([features[labels == k].mean(axis=0) for k in range(n_comp)])
    return merged_coords, merged_feats

# --- 3. Multi-Modal Fused Tracker ---
class CellTrack:
    def __init__(self, start_pos_um: np.ndarray, start_feat: np.ndarray, track_id: int, start_t: int):
        self.id = track_id
        self.x = np.array([start_pos_um[0], start_pos_um[1], start_pos_um[2], 0.0, 0.0, 0.0])
        self.P = np.eye(6) * 10.0
        self.F = np.eye(6)
        self.F[0, 3] = self.F[1, 4] = self.F[2, 5] = 1.0
        self.H = np.zeros((3, 6))
        self.H[0, 0] = self.H[1, 1] = self.H[2, 2] = 1.0
        self.R = np.eye(3) * 2.0
        self.Q = np.eye(6) * 0.1
        self.time_since_update = 0
        self.history = [(start_t, start_pos_um.copy())]
        
        self.avg_feat = start_feat.copy()
        self.alpha_feat = 0.20

    def predict(self) -> np.ndarray:
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        self.time_since_update += 1
        return self.x[:3]

    def update(self, measurement_um: np.ndarray, new_feat: np.ndarray, t: int):
        y = measurement_um - (self.H @ self.x)
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        self.P = (np.eye(6) - K @ self.H) @ self.P
        self.time_since_update = 0
        self.history.append((t, self.x[:3].copy()))
        self.avg_feat = (1.0 - self.alpha_feat) * self.avg_feat + self.alpha_feat * new_feat


class VectorKalmanTracker:
    def __init__(self, max_lost=MAX_LOST_FRAMES, match_dist=MATCH_DIST_UM, 
                 lambda_int=LAMBDA_INT, lambda_geom=LAMBDA_GEOM, sigma_spatial=SIGMA_SPATIAL):
        self.tracks = []
        self.dead_tracks = []
        self.next_id = 1
        self.max_lost = max_lost
        self.match_dist = match_dist
        
        self.lambda_int = lambda_int
        self.lambda_geom = lambda_geom
        self.sigma_spatial = sigma_spatial
        self.sigma_int = SIGMA_INT
        self.sigma_geom = SIGMA_GEOM

    def step(self, detection_dict: dict, t: int):
        detections_um = detection_dict["coords"]
        detections_feat = detection_dict["features"]

        if len(self.tracks) == 0:
            for i, det in enumerate(detections_um):
                self.tracks.append(CellTrack(det, detections_feat[i], self.next_id, t))
                self.next_id += 1
            return

        preds = np.array([trk.predict() for trk in self.tracks])

        if len(detections_um) == 0:
            self._cull()
            return

        # 1. Base spatial distance matrix
        cost_spatial = cdist(preds, detections_um)
        in_spatial_gate = cost_spatial <= self.match_dist

        # 2. Chi-squared Spatial Cost
        z_spatial = (cost_spatial / self.sigma_spatial) ** 2

        # 3. Dual Chi-squared Feature Costs
        if len(detections_feat) > 0 and len(self.tracks) > 0:
            track_feats = np.array([trk.avg_feat for trk in self.tracks])

            # Feature A: Photometric (mean_intensity)
            t_int = track_feats[:, 0, None]
            d_int = detections_feat[None, :, 0]
            rel_int_diff = np.abs(t_int - d_int) / (np.abs(t_int) + np.abs(d_int) + 1e-6)
            z_int = (rel_int_diff / self.sigma_int) ** 2

            # Feature B: Geometric 3D Semi-Axes [a, b, c]
            t_axes = track_feats[:, None, 1:4]
            d_axes = detections_feat[None, :, 1:4]
            axis_diff = np.linalg.norm(t_axes - d_axes, axis=2)
            axis_norm = np.linalg.norm(t_axes, axis=2) + np.linalg.norm(d_axes, axis=2) + 1e-6
            rel_geom_diff = axis_diff / axis_norm
            z_geom = (rel_geom_diff / self.sigma_geom) ** 2

            fused_cost = z_spatial + (self.lambda_int * z_int) + (self.lambda_geom * z_geom)
        else:
            fused_cost = z_spatial

        fused_cost[~in_spatial_gate] = 1e6

        r_idx, c_idx = linear_sum_assignment(fused_cost)
        unmatched_dets = set(range(len(detections_um)))

        for r, c in zip(r_idx, c_idx):
            if in_spatial_gate[r, c]:
                self.tracks[r].update(detections_um[c], detections_feat[c], t)
                unmatched_dets.discard(c)

        for c in unmatched_dets:
            self.tracks.append(CellTrack(detections_um[c], detections_feat[c], self.next_id, t))
            self.next_id += 1

        self._cull()

    def _cull(self):
        alive, dead = [], []
        for trk in self.tracks:
            (dead if trk.time_since_update > self.max_lost else alive).append(trk)
        self.dead_tracks.extend(dead)
        self.tracks = alive

    def all_tracks(self):
        return self.tracks + self.dead_tracks

    def to_networkx(self) -> nx.DiGraph:
        g = nx.DiGraph()
        for trk in self.all_tracks():
            if len(trk.history) < MIN_HITS:
                continue

            hist = sorted(trk.history, key=lambda x: x[0])
            dense_nodes = []

            # Linear gap interpolation (strict Delta t = 1)
            for i in range(len(hist) - 1):
                t1, p1 = hist[i]
                t2, p2 = hist[i + 1]
                dense_nodes.append((t1, p1))
                dt = t2 - t1
                if 1 < dt <= (self.max_lost + 1):
                    for step in range(1, dt):
                        alpha = step / dt
                        dense_nodes.append((t1 + step, (1.0 - alpha) * p1 + alpha * p2))
            dense_nodes.append(hist[-1])

            node_ids = []
            for t_val, pos in dense_nodes:
                nid = f"T{t_val}_ID{trk.id}"
                if not g.has_node(nid):
                    g.add_node(nid, t=int(t_val), z=float(pos[0]), y=float(pos[1]), x=float(pos[2]))
                node_ids.append(nid)

            for u, v in zip(node_ids[:-1], node_ids[1:]):
                g.add_edge(u, v)

        return g


# --- 4. Main Inference Loop Across Test Samples ---
test_zarrs = sorted(glob.glob(os.path.join(test_dir, "*.zarr")))
test_datasets = [os.path.splitext(os.path.basename(p))[0] for p in test_zarrs]

print(f"Discovered {len(test_datasets)} test datasets in {test_dir}")
assert len(test_datasets) > 0, "No test datasets found! Check test_dir path."

submission_rows = []
global_id = 0

for dataset_name in tqdm(test_datasets, desc="Running Test Inference"):
    zarr_path = os.path.join(test_dir, f"{dataset_name}.zarr")
    z_arr = zarr.open(zarr_path, mode="r")['0']
    n_frames = z_arr.shape[0]

    tracker = VectorKalmanTracker()

    for t in range(n_frames):
        raw_frame = np.asarray(z_arr[t])
        det_dict = detect_frame(raw_frame)
        tracker.step(det_dict, t)

    nx_graph = tracker.to_networkx()

    # Convert tracks into dataset-specific node & edge rows
    node_id_counter = 0
    node_id_map = {}
    node_rows = []
    edge_rows = []

    # Sort nodes by time to ensure orderly node_id sequencing
    for node, d in sorted(nx_graph.nodes(data=True), key=lambda x: (x[1]["t"], x[0])):
        z_vox = int(round(float(d["z"]) / SCALE_ZYX[0]))
        y_vox = int(round(float(d["y"]) / SCALE_ZYX[1]))
        x_vox = int(round(float(d["x"]) / SCALE_ZYX[2]))

        nid = node_id_counter
        node_id_map[node] = nid
        node_id_counter += 1

        node_rows.append({
            "dataset": dataset_name,
            "row_type": "node",
            "node_id": nid,
            "t": int(d["t"]),
            "z": z_vox,
            "y": y_vox,
            "x": x_vox,
            "source_id": -1,
            "target_id": -1
        })

    # Add edges between consecutive nodes (Delta t == 1)
    for u, v in nx_graph.edges():
        edge_rows.append({
            "dataset": dataset_name,
            "row_type": "edge",
            "node_id": -1,
            "t": -1,
            "z": -1,
            "y": -1,
            "x": -1,
            "source_id": node_id_map[u],
            "target_id": node_id_map[v]
        })

    # Rows per dataset: all node rows first, then all edge rows
    dataset_rows = node_rows + edge_rows
    for r in dataset_rows:
        r["id"] = global_id
        global_id += 1
        submission_rows.append(r)

    # Free GPU VRAM
    cp.get_default_memory_pool().free_all_blocks()

# --- 5. Save and Verify CSV ---
cols = ["id", "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
df_sub = pd.DataFrame(submission_rows)[cols]
df_sub.to_csv(output_csv, index=False)
print(f"\nWrote {len(df_sub)} rows to {output_csv}")

# --- 6. Built-in Submission Sanity Auditor ---
print("\nRunning submission integrity audit...")
assert os.path.exists(output_csv), "File submission.csv does not exist!"
assert list(df_sub.columns) == cols, f"Columns mismatch: {list(df_sub.columns)} != {cols}"
assert (df_sub["id"].values == np.arange(len(df_sub))).all(), "ID column is not strictly contiguous 0..N-1!"

datasets_in_csv = set(df_sub["dataset"].unique())
assert datasets_in_csv == set(test_datasets), "Missing test datasets in submission!"

# Audit graph properties per dataset
for ds in datasets_in_csv:
    part = df_sub[df_sub["dataset"] == ds]
    n_nodes = (part["row_type"] == "node").sum()
    n_edges = (part["row_type"] == "edge").sum()
    assert n_nodes > 0, f"Dataset {ds} has 0 predicted nodes!"
    assert n_edges > 0, f"Dataset {ds} has 0 predicted edges!"

    valid_nids = set(part[part["row_type"] == "node"]["node_id"])
    edges = part[part["row_type"] == "edge"]
    assert set(edges["source_id"]).issubset(valid_nids), f"Dataset {ds} has invalid source_id!"
    assert set(edges["target_id"]).issubset(valid_nids), f"Dataset {ds} has invalid target_id!"

    # Verify in-degree <= 1
    assert edges["target_id"].value_counts().max() <= 1, f"Dataset {ds} violates in-degree <= 1!"